In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/Air_Quality.csv")
df.head()


,Unique ID,Indicator ID,Name,Measure,Measure Info,Geo Type Name,Geo Join ID,Geo Place Name,Time Period,Start_Date,Data Value,Message
0,742475,365,Fine particles (PM 2.5),Mean,mcg/m3,CD,304,Bushwick (CD4),Summer 2021,06/01/2021,8.778826,NaN
1,877252,375,Nitrogen dioxide (NO2),Mean,ppb,CD,412,Jamaica and Hollis (CD12),Winter 2022-23,12/01/2022,19.657097,NaN
2,743686,386,Ozone (O3),Mean,ppb,UHF42,503,Willowbrook,Summer 2021,06/01/2021,29.086208,NaN
3,825986,375,Nitrogen dioxide (NO2),Mean,ppb,UHF42,502,Stapleton - St. George,Winter 2021-22,12/01/2021,18.562443,NaN
4,643406,375,Nitrogen dioxide (NO2),Mean,ppb,UHF42,402,West Queens,Summer 2019,06/01/2019,15.580000,NaN


We decided to use the New York City air quality dataset to create a semi-supervised model that can predict what type of pollutant it is when this information is not available.

In [16]:
df.shape




(18862, 12)

In [17]:
df.columns

Index(['Unique ID', 'Indicator ID', 'Name', 'Measure', 'Measure Info',
       'Geo Type Name', 'Geo Join ID', 'Geo Place Name', 'Time Period',
       'Start_Date', 'Data Value', 'Message'],
      dtype='object')

In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18862 entries, 0 to 18861
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Unique ID       18862 non-null  int64  
 1   Indicator ID    18862 non-null  int64  
 2   Name            18862 non-null  object 
 3   Measure         18862 non-null  object 
 4   Measure Info    18862 non-null  object 
 5   Geo Type Name   18862 non-null  object 
 6   Geo Join ID     18862 non-null  int64  
 7   Geo Place Name  18862 non-null  object 
 8   Time Period     18862 non-null  object 
 9   Start_Date      18862 non-null  object 
 10  Data Value      18862 non-null  float64
 11  Message         0 non-null      float64
dtypes: float64(2), int64(3), object(7)
memory usage: 1.7+ MB


We decide to aim the model to the column name, the model would predice what type of pollutant.

In [21]:
df_clean = df.drop(columns=[
    "Unique ID",
    "Geo Join ID",
    "Message"
])

We delete this collumns because this are not important for the model

In [22]:

df_clean.head()


,Indicator ID,Name,Measure,Measure Info,Geo Type Name,Geo Place Name,Time Period,Start_Date,Data Value
0,365,Fine particles (PM 2.5),Mean,mcg/m3,CD,Bushwick (CD4),Summer 2021,06/01/2021,8.778826
1,375,Nitrogen dioxide (NO2),Mean,ppb,CD,Jamaica and Hollis (CD12),Winter 2022-23,12/01/2022,19.657097
2,386,Ozone (O3),Mean,ppb,UHF42,Willowbrook,Summer 2021,06/01/2021,29.086208
3,375,Nitrogen dioxide (NO2),Mean,ppb,UHF42,Stapleton - St. George,Winter 2021-22,12/01/2021,18.562443
4,375,Nitrogen dioxide (NO2),Mean,ppb,UHF42,West Queens,Summer 2019,06/01/2019,15.580000


The columns above are the ones we will use to train the model, keeping in mind that the aim column will be the target column.

In [23]:
df_clean.columns

Index(['Indicator ID', 'Name', 'Measure', 'Measure Info', 'Geo Type Name',
       'Geo Place Name', 'Time Period', 'Start_Date', 'Data Value'],
      dtype='object')

In [24]:
df_clean["Start_Date"] = pd.to_datetime(df_clean["Start_Date"])

First, we convert Start_Date to date format. This is in order to extract important information.

In [25]:
df_clean["year"] = df_clean["Start_Date"].dt.year
df_clean["month"] = df_clean["Start_Date"].dt.month

In [26]:
df_clean = df_clean.drop(columns=["Start_Date"])

Now we proceed to separate the variables (x) which are the predictor variables and (y) which is the class we want to predict

In [27]:
X = df_clean.drop(columns=["Name"])
y = df_clean["Name"]

    Converting categorical variables

Now we apply One Hot Encoding.  

In [28]:
X = pd.get_dummies(X)

This converts variables such as:

    Geo Place Name
    Measure
    Time Period

into binary variables for the model.

In [29]:
X.shape

(18862, 196)

In [30]:
X.head()

,Indicator ID,Data Value,year,month,Measure_Annual average concentration,Measure_Estimated annual rate,Measure_Estimated annual rate (age 18+),Measure_Estimated annual rate (age 30+),Measure_Estimated annual rate (under age 18),Measure_Mean,...,Time Period_Winter 2013-14,Time Period_Winter 2014-15,Time Period_Winter 2015-16,Time Period_Winter 2016-17,Time Period_Winter 2017-18,Time Period_Winter 2018-19,Time Period_Winter 2019-20,Time Period_Winter 2020-21,Time Period_Winter 2021-22,Time Period_Winter 2022-23
0,365,8.778826,2021,6,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
1,375,19.657097,2022,12,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,True
2,386,29.086208,2021,6,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
3,375,18.562443,2021,12,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,True,False
4,375,15.580000,2019,6,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False


In [31]:
from sklearn.model_selection import train_test_split

First we separate the evaluation data.

In [32]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Now we simulate few labels.

In [33]:
X_labeled, X_unlabeled, y_labeled, y_unlabeled = train_test_split(
    X_train,
    y_train,
    test_size=0.8,
    random_state=42,
    stratify=y_train
)

Check sizes

In [34]:
print("Labeled:", X_labeled.shape)
print("Unlabeled:", X_unlabeled.shape)
print("Test:", X_test.shape)

Labeled: (3017, 196)
Unlabeled: (12072, 196)
Test: (3773, 196)


Saving the Datasets (Reproducibility)

In [35]:
import joblib

In [36]:
joblib.dump(X_labeled, "../data/X_labeled.pkl")
joblib.dump(y_labeled, "../data/y_labeled.pkl")

joblib.dump(X_unlabeled, "../data/X_unlabeled.pkl")

joblib.dump(X_test, "../data/X_test.pkl")
joblib.dump(y_test, "../data/y_test.pkl")

['../data/y_test.pkl']

Test that they can be loaded

In [37]:
X_labeled = joblib.load("../data/X_labeled.pkl")

print(X_labeled.shape)

(3017, 196)
